# Inspect the least-squares experiments

Read-only analysis of an existing run: **no simulations, job submissions, or result overwrites**.
Select the `FL_EF21_ENV` kernel, then run the cells in order. The default run below is job **54712913**.

- **A–B:** actual sampled initialization; iterations, not evaluation budgets.
- **C–D:** prescribed initialization to isolate mechanisms; no acquisition-cost comparison.

The `.err` log containing seven passing tests is normal: Python's unittest runner writes its report to stderr.


In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt

# Find experiments/ whether the notebook starts here or at a repository root.
candidates = [
    parent / suffix
    for parent in [Path.cwd(), *Path.cwd().parents]
    for suffix in (Path("."), Path("experiments"), Path("TSGD/experiments"))
]
EXPERIMENT_DIR = next(
    (p.resolve() for p in candidates if (p / "least_squares.py").is_file()), None
)
if EXPERIMENT_DIR is None:
    raise FileNotFoundError("Open this notebook from the experiments folder or its repository.")

# EDIT THIS LINE to select another job, or use an absolute path to downloaded results.
RUN_DIR = EXPERIMENT_DIR / "results" / "cluster" / "full_54712913"
# For the earlier local run: RUN_DIR = EXPERIMENT_DIR / "results"

available = sorted((EXPERIMENT_DIR / "results").rglob("settings_and_summary.json"))
print("Available result folders:")
for path in available:
    print(" ", path.parent.relative_to(EXPERIMENT_DIR))
print("\nSelected:", RUN_DIR)

for name in ("curves.csv", "settings_and_summary.json"):
    if not (RUN_DIR / name).is_file():
        raise FileNotFoundError(
            f"Missing {RUN_DIR / name}. Edit RUN_DIR above; no other run is selected silently."
        )


In [ ]:
import sys
if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))
from plot_least_squares import load_results, make_figure, draw_panel

series, settings = load_results(RUN_DIR)
specs = {(s["panel"], s["name"]): s for s in settings["scenarios"]}
print(json.dumps({key: settings.get(key) for key in (
    "runs", "steps", "dimension", "eta", "seed", "python_version",
    "numpy_version", "matplotlib_version", "runtime_environment", "comparison"
)}, indent=2))
print(f"Loaded {len(series)} scenarios, {sum(map(len, series.values()))} iteration records.")


## Four independent plots

Each plot has one legend, no inset, and no explanatory footer. Initialization assumptions stay in the text here: A–B use sampled initialization; C–D use controlled initialization. All comparisons use iterations, not budgets.


In [ ]:
OVERVIEW_MAX_ITER = 80
for panel in "ABCD":
    fig = make_figure(series, settings, panel, max_iter=OVERVIEW_MAX_ITER)
    plt.show()


## Plot controls and helpers

Edit `SHOW_EXACT`, `SHOW_BANDS`, or `MAX_ITER`, then rerun the desired panel. Solid/dashed colored curves are Monte Carlo means; dotted curves are **exact expectations**, not convergence upper bounds. Bands are ±2 pointwise Monte Carlo standard errors, not simultaneous confidence bands.


In [ ]:
SHOW_EXACT = False  # True overlays dotted exact finite-time expectations.
SHOW_BANDS = True
MAX_ITER = 80  # None shows every iteration; y-limits fit the visible data.

def table(headers, rows):
    formatted = [[f"{v:.5g}" if isinstance(v, (float, np.floating)) else str(v)
                  for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(row[i]) for row in formatted))
              for i, h in enumerate(headers)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in formatted:
        print(" | ".join(v.ljust(w) for v, w in zip(row, widths)))

def column(key, field):
    return np.array([row[field] for row in series[key]])


def plot_panel(panel, metric="gradient"):
    fig, ax = plt.subplots(figsize=(10, 5.5))
    draw_panel(ax, panel, series, settings, max_iter=MAX_ITER, metric=metric,
               exact=SHOW_EXACT, bands=SHOW_BANDS)
    if SHOW_EXACT:
        fig.suptitle("Dotted curves: exact finite-time expectations", fontsize=10)
    fig.tight_layout()
    plt.show()


## A — Accurate differences despite noisy gradients

Actual initial batches contain 1, 128, or 512 samples. The table reports expected and observed initialization MSE without crowding the plot legend. Accurate transport preserves poor initial information too. These are mechanism demonstrations, not computational savings comparisons.


In [ ]:
table(["Method", "Initialization", "Batch", "Expected E0", "Observed E0"],
      [[s["name"], s["initialization"], s["batch"], s["expected_E0"],
        series[key][0]["tracking_mean"]]
       for key, s in specs.items() if key[0] == "A"])
plot_panel("A")


## B — SGD shines and TSGD falsely converges

Compare TSGD's true-gradient squared norm with its estimator squared norm. Black markers identify SGD's true gradient; the orange dashed curve is TSGD's nearly coincident estimator. These are different quantities, not duplicated data.


In [ ]:
plot_panel("B")
table(["Method", "Final true gradient", "Final estimator", "Final tracking error"],
      [[key[1], rows[-1]["gradient_mean"], rows[-1]["estimator_mean"],
        rows[-1]["tracking_mean"]]
       for key, rows in series.items() if key[0] == "B"])


## C — Additive noise cancels

The label-noise levels share a prescribed initial estimate and coupled feature samples. TSGD curves coincide up to roundoff. **The remaining plateau is due to stochastic transport (δ=0.2), not label noise.** Markers distinguish coincident curves; one legend entry groups them. Label noise would affect sampled initialization, which this panel intentionally holds fixed.


In [ ]:
plot_panel("C")
keys_c = [key for key, s in specs.items() if key[0] == "C" and s["method"] == "TSGD"]
reference = column(keys_c[0], "gradient_mean")
table(["TSGD setting", "Max difference between saved mean curves"],
      [[key[1], float(np.max(np.abs(column(key, "gradient_mean") - reference)))]
       for key in keys_c])
print("Run-time cancellation diagnostic:", settings["max_cancellation_difference"])
print("This recorded diagnostic includes final x/g differences and differences between mean curves.")
print("Individual trajectory histories were not saved in the CSV.")


## D — Initialization quality × similarity

A single legend identifies each (δ, E₀) combination. Color distinguishes δ and line style distinguishes E₀. Population curvature and stepsize are fixed. The initial errors are prescribed, not acquired at comparable sampling cost. Large transport noise limits the benefit of improving initialization.


In [ ]:
plot_panel("D")


## Final errors versus exact expectations

`Exact at K` is the exact finite-time expectation. `Limit` is the constant-stepsize asymptotic prediction; finite-time errors need not equal it. The standardized discrepancy `(MC − exact)/SE` is descriptive, **not** a collection of independent hypothesis tests: curves share randomness and iterations are correlated. Zero SE is reported as undefined.


In [ ]:
comparison_rows = []
for key, spec in specs.items():
    row = series[key][-1]
    mean, exact, se = (row["gradient_mean"], row["gradient_exact"], row["gradient_se"])
    z = (mean - exact) / se if se > 0 else float("nan")
    comparison_rows.append([key[0], key[1], mean, se, exact,
                            spec["limiting_gradient_exact"], z])
table(["Panel", "Scenario", "MC at K", "SE", "Exact at K", "Limit", "(MC-exact)/SE"],
      comparison_rows)


## Optional detailed view

Change the panel, metric (`gradient`, `estimator`, or `tracking`), and iteration range. This only redraws stored data; it does not change the simulation parameters.


In [ ]:
MAX_ITER = 80
plot_panel("D", metric="tracking")
